<a href="https://colab.research.google.com/github/yiding2022/DAAI/blob/main/KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
import matplotlib.pylab as plt

In [ ]:
from google.colab import files

# Upload file
uploaded = files.upload()


Saving RidingMowers.txt to RidingMowers (2).txt


In [4]:
url = "https://raw.githubusercontent.com/yiding2022/DAAI/main/RidingMowers.txt"
mower_df = pd.read_csv(url)

In [5]:
mower_df
# lot size in thousands of sq. feet

,Income,Lot_Size,Ownership
0,60.0,18.4,Owner
1,85.5,16.8,Owner
2,64.8,21.6,Owner
3,61.5,20.8,Owner
4,87.0,23.6,Owner
5,110.1,19.2,Owner
6,108.0,17.6,Owner
7,82.8,22.4,Owner
8,69.0,20.0,Owner
9,93.0,20.8,Owner


In [ ]:
# generate ID number for each observation
mower_df['Number'] = mower_df.index + 1

In [ ]:
# splitting into training and testing sets - critical for supervised learning algorithms
trainData, validData = train_test_split(mower_df, test_size=0.4, random_state=26)


In [ ]:
## new household - which we want to classify into one of the two classes
newHousehold = pd.DataFrame([{'Income': 60, 'Lot_Size': 20}])


In [ ]:
mower_df

,Income,Lot_Size,Ownership,Number
0,60.0,18.4,Owner,1
1,85.5,16.8,Owner,2
2,64.8,21.6,Owner,3
3,61.5,20.8,Owner,4
4,87.0,23.6,Owner,5
5,110.1,19.2,Owner,6
6,108.0,17.6,Owner,7
7,82.8,22.4,Owner,8
8,69.0,20.0,Owner,9
9,93.0,20.8,Owner,10


In [ ]:
predictors = ['Income', 'Lot_Size']
outcome = 'Ownership'


In [ ]:

# initialize the knn model
# initialize normalized training, validation, and complete data frames
# use the training data to learn the transformation.
scaler = preprocessing.StandardScaler()
scaler.fit(trainData[['Income', 'Lot_Size']]) # Note use of array of column names


StandardScaler()

In [ ]:
# Transform the full dataset
# REMEMBER! we always use training data to learn the transformation, and used the leaned transformation to transform all data
# concat allows us to join the dataframes together

mowerNorm = pd.concat([pd.DataFrame(scaler.transform(mower_df[['Income','Lot_Size']]), columns=['zIncome', 'zLot_Size']),mower_df[['Ownership', 'Number']]], axis=1)



In [ ]:
mowerNorm

,zIncome,zLot_Size,Ownership,Number
0,-0.477910,-0.174908,Owner,1
1,0.680365,-0.787085,Owner,2
2,-0.259882,1.049447,Owner,3
3,-0.409776,0.743358,Owner,4
4,0.748499,1.814668,Owner,5
5,1.797760,0.131181,Owner,6
6,1.702373,-0.480996,Owner,7
7,0.557724,1.355535,Owner,8
8,-0.069107,0.437269,Owner,9
9,1.021034,0.743358,Owner,10


In [ ]:
trainNorm = mowerNorm.iloc[trainData.index]
validNorm = mowerNorm.iloc[validData.index]
newHouseholdNorm = pd.DataFrame(scaler.transform(newHousehold), columns=['zIncome', 'zLot_Size'])

In [ ]:
# use NearestNeighbors from scikit-learn to compute knn
from sklearn.neighbors import NearestNeighbors
knn = NearestNeighbors(n_neighbors=3) # find the 3 nearest neighbors
knn.fit(trainNorm.iloc[:, 0:2])
distances, indices = knn.kneighbors(newHouseholdNorm)


In [ ]:
# indices is a list of lists, we are only interested in the first element
# the outcome are the three neasrest neighbors
# based on the majority class of the three nearest neighbors, we can classify the new household as owner.
trainNorm.iloc[indices[0], :]

,zIncome,zLot_Size,Ownership,Number
3,-0.409776,0.743358,Owner,4
13,-0.804953,0.743358,Nonowner,14
0,-0.477910,-0.174908,Owner,1


Code for choosing k

In [ ]:
# training and validation data

train_X = trainNorm[['zIncome', 'zLot_Size']]
train_y = trainNorm['Ownership']
valid_X = validNorm[['zIncome', 'zLot_Size']]
valid_y = validNorm['Ownership']

In [ ]:
# Train a classifier for different values of k
results = []
for k in range(1, 15):
    knn = KNeighborsClassifier(n_neighbors=k).fit(train_X, train_y)
    results.append({
        'k': k,
        'accuracy': accuracy_score(valid_y, knn.predict(valid_X))
    })

# Convert results to a pandas data frame
results = pd.DataFrame(results)
print(results)
# k=4 is the best value for k

     k  accuracy
0    1       0.6
1    2       0.7
2    3       0.8
3    4       0.9
4    5       0.7
5    6       0.9
6    7       0.9
7    8       0.9
8    9       0.9
9   10       0.8
10  11       0.8
11  12       0.9
12  13       0.4
13  14       0.4
